# 08 – Silver History: DESCRIBE HISTORY, Time Travel, CDF, RESTORE

Demonstrates Delta Lake features on the silver tables.

**No reset.** Reads the state left by 07_silver_merge_edge_cases.

| Section | Content |
|---|---|
| 1 | Guard: batch_log has 12 SUCCESS rows |
| 2 | DESCRIBE HISTORY: card in detail, other tables in summary, operationMetrics ↔ batch_log |
| 3 | Time travel: card row count per version, client 435 before / after, TIMESTAMP AS OF example |
| 4 | CDF: cancelled cards, batch_log check per batch, comparison with landing records |
| 5 | RESTORE: back to batch 1, then forward to the last MERGE version |

⚠️ Time travel depends on old data files. If VACUUM has run (7-day retention), old versions may not be readable. In that case, run 07 first, then 08.

## 1. Guard

08 does not reset, so batches 1 → 3 must already be applied to the 4 tables (4 × 3 = 12 SUCCESS).

Expected output: "Guard OK" and the batch_log table. If it fails, run 07 first.

In [0]:
from pyspark.sql import functions as F

EXPECTED_SUCCESS = 12

df_log = spark.table("workspace.silver.batch_log")
success_count = df_log.filter(F.col("status") == "SUCCESS").count()

# 08 does not reset; if the batches are not applied, later sections give wrong results, so stop here
if success_count != EXPECTED_SUCCESS:
    raise Exception(f"batch_log has {success_count} SUCCESS rows, expected {EXPECTED_SUCCESS}. Run 07_silver_merge_edge_cases first.")

print("Guard OK: SUCCESS =", success_count)
display(df_log.orderBy("batch_id", "table_name"))

## 2. DESCRIBE HISTORY

Delta stores every write as a version. DESCRIBE HISTORY lists them: version, timestamp, operation, operationMetrics.

After the last reset (CREATE OR REPLACE) version, the MERGEs are batch 1, 2, 3 in order. Version numbers are not hard-coded; they are read from the history.

Card has all three operations (I / U / D), so it is shown in detail.

Expected output: 3 rows. Expected inserted / updated / deleted: batch 1 = 2 / 7 / 4, batch 2 = 5 / 8 / 2, batch 3 = 1 / 6 / 4.

In [0]:
from pyspark.sql import Window

def get_merge_history(table_name):
    full_name = "workspace.silver." + table_name
    df_history = spark.sql("DESCRIBE HISTORY " + full_name)

    # Last reset = last CREATE OR REPLACE version (reset_silver rewrites the table with overwrite)
    df_resets = df_history.filter(F.col("operation").startswith("CREATE OR REPLACE"))
    reset_version = df_resets.agg(F.max("version")).collect()[0][0]
    print(table_name, "reset_version =", reset_version)

    # MERGEs only; automatic operations such as serverless OPTIMIZE are excluded
    df_merges = df_history.filter((F.col("operation") == "MERGE") & (F.col("version") > reset_version))

    # 1st, 2nd, 3rd MERGE after reset = batch 1, 2, 3
    window_by_version = Window.orderBy("version")
    df_merges = df_merges.withColumn("batch_id", F.row_number().over(window_by_version))

    df_result = df_merges.select(
        F.lit(table_name).alias("table_name"),
        "batch_id",
        "version",
        "timestamp",
        "operation",
        F.col("operationMetrics")["numTargetRowsInserted"].cast("int").alias("rows_inserted"),
        F.col("operationMetrics")["numTargetRowsUpdated"].cast("int").alias("rows_updated"),
        F.col("operationMetrics")["numTargetRowsDeleted"].cast("int").alias("rows_deleted"),
        "operationMetrics",
    )
    return df_result

df_card_history = get_merge_history("card")
display(df_card_history.orderBy("version"))

### 2b. Summary: 4 tables vs batch_log

For each table, the 3 MERGE versions after reset: Delta's operationMetrics counts compared with the counts in batch_log.

Expected output: 12 rows. history_* columns come from Delta, log_* columns from batch_log. ok should be true in all 12 rows.

In [0]:
CDC_TABLES = ["client", "account", "card", "loan"]

df_all_history = None
for table_name in CDC_TABLES:
    df_one = get_merge_history(table_name).drop("operationMetrics")
    if df_all_history is None:
        df_all_history = df_one
    else:
        df_all_history = df_all_history.unionByName(df_one)

df_history_counts = df_all_history.select(
    "table_name",
    "batch_id",
    "version",
    F.col("rows_inserted").alias("history_inserted"),
    F.col("rows_updated").alias("history_updated"),
    F.col("rows_deleted").alias("history_deleted"),
)

df_log_counts = df_log.select(
    "table_name",
    "batch_id",
    F.col("rows_inserted").alias("log_inserted"),
    F.col("rows_updated").alias("log_updated"),
    F.col("rows_deleted").alias("log_deleted"),
)

# Do Delta's own counts and our log describe the same MERGE?
df_compare = df_history_counts.join(df_log_counts, on=["table_name", "batch_id"], how="left")

is_ok = (
    (F.col("history_inserted") == F.col("log_inserted"))
    & (F.col("history_updated") == F.col("log_updated"))
    & (F.col("history_deleted") == F.col("log_deleted"))
)
df_compare = df_compare.withColumn("ok", is_ok)

display(df_compare.orderBy("table_name", "batch_id"))
print("ok rows:", df_compare.filter(F.col("ok")).count(), "/ 12")

## 3. Time travel

Delta keeps old versions. With VERSION AS OF, a table can be read as it was in the past.

Versions are used instead of timestamps: the runner applies the batches a few seconds apart, so timestamps are fragile.

### 3a. Card: row count per version

Versions are read from the history. The version right before batch 1 is the state after reset (after_batch = 0).

Expected output: 4 rows. Expected row_count: 892 → 890 → 893 → 890.

In [0]:
card_merge_rows = df_card_history.orderBy("version").select("batch_id", "version").collect()

# Version right before batch 1 = state after reset (a CDF ALTER TABLE may sit in between; the data is the same)
first_merge_version = card_merge_rows[0]["version"]
versions_to_check = [(0, first_merge_version - 1)]
for row in card_merge_rows:
    versions_to_check.append((row["batch_id"], row["version"]))

results = []
for batch_id, version in versions_to_check:
    query = f"SELECT COUNT(*) AS row_count FROM workspace.silver.card VERSION AS OF {version}"
    row_count = spark.sql(query).collect()[0]["row_count"]
    results.append((batch_id, version, row_count))

df_card_versions = spark.createDataFrame(results, ["after_batch", "version", "row_count"])
display(df_card_versions.orderBy("version"))

### 3b. Client 435: before and after batch 2

In batch 2, client 435 moved twice (U → U): first to district 32, then to 28. Dedup kept the last record.

Before batch 2 = batch 2 MERGE version - 1. This client did not change in batch 1, so the "before" value is the bronze value.

Expected output: the before row has the same district_id as bronze and _change_ts 1998-12-31; the after row has district_id 28 and _change_ts 1999-02-18. The intermediate value 32 does not appear in any version.

In [0]:
df_client_history = get_merge_history("client")
batch2_version = df_client_history.filter(F.col("batch_id") == 2).collect()[0]["version"]
before_version = batch2_version - 1

query_before = f"SELECT client_id, district_id, _change_ts FROM workspace.silver.client VERSION AS OF {before_version} WHERE client_id = 435"
query_after = f"SELECT client_id, district_id, _change_ts FROM workspace.silver.client VERSION AS OF {batch2_version} WHERE client_id = 435"

df_before = spark.sql(query_before).withColumn("state", F.lit("before batch 2")).withColumn("version", F.lit(before_version))
df_after = spark.sql(query_after).withColumn("state", F.lit("after batch 2")).withColumn("version", F.lit(batch2_version))

# Every column is a string in bronze; printed only for comparison
df_bronze_435 = spark.table("workspace.bronze.client").filter(F.col("client_id") == "435")
bronze_district = df_bronze_435.collect()[0]["district_id"]
print("bronze district_id:", bronze_district)

display(df_before.unionByName(df_after))

### 3c. TIMESTAMP AS OF (single example)

Time travel also works with a timestamp: the version of the last commit at or before that time is read.

The timestamp is read from the history (card batch 2 MERGE). VERSION AS OF is the main method in this project; this cell only shows the alternative.

Expected output: row_count 893 (same as the batch 2 row in 3a).

In [0]:
df_card_batch2 = df_card_history.filter(F.col("batch_id") == 2)

# Format the timestamp as a string inside Spark; converting it to a Python datetime can shift the timezone
df_card_batch2 = df_card_batch2.withColumn("ts_text", F.date_format("timestamp", "yyyy-MM-dd HH:mm:ss.SSS"))
batch2_row = df_card_batch2.select("version", "ts_text").collect()[0]

query = f"SELECT COUNT(*) AS row_count FROM workspace.silver.card TIMESTAMP AS OF '{batch2_row['ts_text']}'"
row_count = spark.sql(query).collect()[0]["row_count"]

print("batch 2 version  :", batch2_row["version"])
print("batch 2 timestamp:", batch2_row["ts_text"])
print("row_count        :", row_count)

## 4. Change Data Feed (CDF)

On a CDF-enabled table, Delta records every change at row level. It is read with table_changes().

_change_type: insert, update_preimage (before), update_postimage (after), delete.

_commit_version shows the version (i.e. the batch) in which the row changed.

Start version = first MERGE version after reset (read from the history).

### 4a. Cancelled cards

Card cancellation is a hard delete in silver. Deleted cards are no longer in silver, but CDF keeps their last state.

Expected output: cards with _change_type = delete. Expected total: 10 (batch 1 = 4, batch 2 = 2, batch 3 = 4).

In [0]:
# First MERGE after reset; intermediate versions such as the reset and CDF ALTER TABLE are excluded
start_version = df_card_history.agg(F.min("version")).collect()[0][0]
print("CDF start_version:", start_version)

# End = last MERGE; later RESTORE versions must not mix into the CDF counts
end_version = df_card_history.agg(F.max("version")).collect()[0][0]
print("CDF end_version:", end_version)

query = f"SELECT * FROM table_changes('workspace.silver.card', {start_version}, {end_version})"
df_card_changes = spark.sql(query)

# _commit_version → batch_id mapping from the history
df_version_to_batch = df_card_history.select(F.col("version").alias("_commit_version"), "batch_id")
df_card_changes = df_card_changes.join(df_version_to_batch, on="_commit_version", how="left")

df_cancelled_cards = df_card_changes.filter(F.col("_change_type") == "delete")

display(
    df_cancelled_cards
    .select("batch_id", "_commit_version", "card_id", "disp_id", "type", "issued_date", "_commit_timestamp")
    .orderBy("batch_id", "card_id")
)
print("Cancelled cards:", df_cancelled_cards.count())

### 4b. CDF ↔ batch_log (card, per batch)

Shows that the row-level change counts in CDF match the counts in batch_log.

Each update produces two CDF rows (update_preimage + update_postimage); only update_postimage is counted.

Expected output: 3 rows, ok true in all 3. Expected ins / upd / del: batch 1 = 2 / 7 / 4, batch 2 = 5 / 8 / 2, batch 3 = 1 / 6 / 4.

In [0]:
change_types = ["insert", "update_postimage", "delete"]

# _change_type values become columns; a type missing in a batch comes as NULL and is set to 0
df_cdf_pivot = df_card_changes.groupBy("batch_id").pivot("_change_type", change_types).count()

df_cdf_counts = df_cdf_pivot.select(
    "batch_id",
    F.coalesce(F.col("insert"), F.lit(0)).alias("cdf_inserted"),
    F.coalesce(F.col("update_postimage"), F.lit(0)).alias("cdf_updated"),
    F.coalesce(F.col("delete"), F.lit(0)).alias("cdf_deleted"),
)

df_card_log_counts = df_log_counts.filter(F.col("table_name") == "card").drop("table_name")

df_cdf_compare = df_cdf_counts.join(df_card_log_counts, on="batch_id", how="left")

is_cdf_ok = (
    (F.col("cdf_inserted") == F.col("log_inserted"))
    & (F.col("cdf_updated") == F.col("log_updated"))
    & (F.col("cdf_deleted") == F.col("log_deleted"))
)
df_cdf_compare = df_cdf_compare.withColumn("ok", is_cdf_ok)

display(df_cdf_compare.orderBy("batch_id"))

### 4c. Landing ↔ CDF + quarantine

How each card record in landing is reflected in silver. Two steps change the counts:

- Quarantine: U / D for a key not in the target (card 101254, batch 3 D).

- Dedup: the U of a key that also has an I in the same batch merges into a single insert (card 1253, batch 2 I → U).

Formula: landing = cdf + quarantine + merged_by_dedup

Expected output: 3 rows (I, U, D), ok true in all 3. Expected U: 22 = 21 + 0 + 1; D: 11 = 10 + 1 + 0.

In [0]:
# Landing card batches (schema inference is enough; we only count)
df_landing_card = spark.read.json("/Volumes/workspace/bronze/landing/card/batch_*/")

landing_i = df_landing_card.filter(F.col("op_type") == "I").count()
landing_u = df_landing_card.filter(F.col("op_type") == "U").count()
landing_d = df_landing_card.filter(F.col("op_type") == "D").count()

# If the same key also has an I in the same batch, its U merges into the I in dedup (I → U rule)
df_landing_inserts = df_landing_card.filter(F.col("op_type") == "I").select("card_id", "batch_id")
df_landing_updates = df_landing_card.filter(F.col("op_type") == "U").select("card_id", "batch_id")
df_u_merged = df_landing_updates.join(df_landing_inserts, on=["card_id", "batch_id"], how="inner")
u_merged_by_dedup = df_u_merged.count()
print("U records merged into I by dedup:", df_u_merged.collect())

df_card_quarantine = spark.table("workspace.silver.quarantine").filter(F.col("table_name") == "card")
quarantine_u = df_card_quarantine.filter(F.col("op_type") == "U").count()
quarantine_d = df_card_quarantine.filter(F.col("op_type") == "D").count()

cdf_i = df_card_changes.filter(F.col("_change_type") == "insert").count()
cdf_u = df_card_changes.filter(F.col("_change_type") == "update_postimage").count()
cdf_d = df_card_changes.filter(F.col("_change_type") == "delete").count()

rows = [
    ("I", landing_i, cdf_i, 0, 0),
    ("U", landing_u, cdf_u, quarantine_u, u_merged_by_dedup),
    ("D", landing_d, cdf_d, quarantine_d, 0),
]
df_landing_compare = spark.createDataFrame(rows, ["op_type", "landing", "cdf", "quarantine", "merged_by_dedup"])

expected_landing = F.col("cdf") + F.col("quarantine") + F.col("merged_by_dedup")
df_landing_compare = df_landing_compare.withColumn("ok", F.col("landing") == expected_landing)

display(df_landing_compare)

## 5. RESTORE

RESTORE returns a table to a past version. Typical scenario: a wrong batch was loaded; how do you roll it back?

Safe flow: card first goes back to the batch 1 version, then forward to the last MERGE version. This way batch_log and silver are consistent at the end.

RESTORE does not delete history; it is added as a new version (operation = RESTORE).

Expected output: after going back to batch 1, 890 rows and no card 1253 (it was added in batch 2). After going forward to the last MERGE, 890 rows and card 1253 gold.

⚠️ In real life, batch_log must also be updated after RESTORE. Otherwise the rolled-back batch still shows SUCCESS and the runner does not run it again.

In [0]:
batch1_version = df_card_history.filter(F.col("batch_id") == 1).collect()[0]["version"]
last_merge_version = df_card_history.agg(F.max("version")).collect()[0][0]
print("batch 1 version:", batch1_version, "| last MERGE version:", last_merge_version)

def show_card_state(label):
    df_card = spark.table("workspace.silver.card")
    row_count = df_card.count()
    rows_1253 = df_card.filter(F.col("card_id") == 1253).collect()
    if len(rows_1253) == 0:
        type_1253 = "none"
    else:
        type_1253 = rows_1253[0]["type"]
    print(label, "| rows:", row_count, "| card 1253:", type_1253)

# 1) Wrong batch scenario: batches 2 and 3 are rolled back
spark.sql(f"RESTORE TABLE workspace.silver.card TO VERSION AS OF {batch1_version}")
show_card_state("after RESTORE to batch 1   ")

# 2) Forward to the last MERGE state; batch_log and silver are consistent again
spark.sql(f"RESTORE TABLE workspace.silver.card TO VERSION AS OF {last_merge_version}")
show_card_state("after RESTORE to last MERGE")

df_card_last_ops = spark.sql("DESCRIBE HISTORY workspace.silver.card").select("version", "timestamp", "operation", "operationParameters")
display(df_card_last_ops.orderBy(F.col("version").desc()).limit(3))